In [10]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, BaggingClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, NuSVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, BaggingClassifier
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from imblearn.over_sampling import SMOTE, SMOTENC

from utils import save_predictions_to_csv, standardize_data, calculate_auc_score, compare_auc_scores
from natsort import natsorted
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from category_encoders import TargetEncoder

In [11]:
dataset_names = []
X_trains = []
y_trains = []
X_tests = []

for folder_name in natsorted(os.listdir("./Competition_data")):
    dataset_names.append(folder_name)
    
    # 讀取原始資料
    X_train = pd.read_csv(f"./Competition_data/{folder_name}/X_train.csv", header=0)
    y_train = pd.read_csv(f"./Competition_data/{folder_name}/y_train.csv", header=0)
    X_test = pd.read_csv(f"./Competition_data/{folder_name}/X_test.csv", header=0)

    # 檢查是否有增強資料
    augment_folder_path = f"./Augmented_data/{folder_name}/"
    
    if os.path.exists(f"{augment_folder_path}X_train_augmented.csv"):
        X_train_aug = pd.read_csv(f"{augment_folder_path}X_train_augmented.csv", header=0)
        y_train_aug = pd.read_csv(f"{augment_folder_path}y_train_augmented.csv", header=0)
        
        # 合併增強資料
        X_train = pd.concat([X_train, X_train_aug], axis=0, ignore_index=True)
        y_train = pd.concat([y_train, y_train_aug], axis=0, ignore_index=True)
    else:
        print(folder_name)

    # 將讀取的資料加入列表
    X_trains.append(X_train)
    y_trains.append(y_train)
    X_tests.append(X_test)

Dataset_26
Dataset_31


In [12]:
def preprocess_data(X_train, y_train, X_test):

    # 區分數值型和類別型特徵
    numeric_features = X_train.select_dtypes(include=['float']).columns
    categorical_features = X_train.select_dtypes(include=['int']).columns

    # 將類別型特徵轉換為索引
    categorical_feature_indices = [X_train.columns.get_loc(col) for col in categorical_features]

    # 類別不平衡處理
    if len(categorical_feature_indices) == 0:
        smote = SMOTE(random_state=40)
        X_train, y_train = smote.fit_resample(X_train, y_train)
    else:
        categorical_feature_indices_bool = [i in categorical_feature_indices for i in range(X_train.shape[1])]
        smote_nc = SMOTENC(categorical_features=categorical_feature_indices_bool, random_state=41)
        X_train, y_train = smote_nc.fit_resample(X_train, y_train)
    
    # 數值型特徵標準化
    numeric_transformer = MinMaxScaler()
    X_train_numeric = numeric_transformer.fit_transform(X_train[numeric_features])
    X_test_numeric = numeric_transformer.transform(X_test[numeric_features])

    # 類別型特徵目標編碼
    if len(categorical_features) > 0:
        categorical_transformer = TargetEncoder(cols=categorical_features)
        X_train_categorical = categorical_transformer.fit_transform(X_train[categorical_features], y_train)
        X_test_categorical = categorical_transformer.transform(X_test[categorical_features])
    else:
        X_train_categorical = np.empty((X_train_numeric.shape[0], 0))
        X_test_categorical = np.empty((X_test_numeric.shape[0], 0))

    # 合併處理後的特徵
    X_train = np.hstack((X_train_numeric, X_train_categorical))
    X_test = np.hstack((X_test_numeric, X_test_categorical))

    return X_train, y_train, X_test

In [15]:
def feature_selection_union(X_train, X_test, y_train):

    # Random Forest
    rf = RandomForestClassifier(n_estimators=400,max_depth=20,random_state=42)
    rf.fit(X_train, y_train)
    rf_importances = rf.feature_importances_
    rf_threshold = np.mean(rf_importances)
    rf_selected_features = np.where(rf_importances >= rf_threshold)[0]

    # XGBoost
    xgb = XGBClassifier(n_estimators=400,max_depth=20,random_state=42, eval_metric='auc')
    xgb.fit(X_train, y_train)
    xgb_importances = xgb.feature_importances_
    xgb_threshold = np.mean(xgb_importances)
    xgb_selected_features = np.where(xgb_importances >= xgb_threshold)[0]

    # 取聯集
    selected_features = np.union1d(rf_selected_features, xgb_selected_features)

    # 篩選特徵
    if isinstance(X_train, pd.DataFrame):
        X_train_selected = X_train.iloc[:, selected_features]
        X_test_selected = X_test.iloc[:, selected_features]
    else:
        X_train_selected = X_train[:, selected_features]
        X_test_selected = X_test[:, selected_features]

    return X_train_selected, X_test_selected

In [17]:
validation_auc_scores = []

for i in range(len(X_trains)):
    X_train, y_train, X_test = preprocess_data(X_trains[i], y_trains[i].values.ravel(), X_tests[i])

    # 使用 train_test_split 進行簡單的訓練/測試拆分
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

    test_predictions_all_folds = []
    fold_auc_scores = []

    # 設定基模型
    base_learners = [
        ('rf', RandomForestClassifier(n_estimators=1000, max_depth=30)),
        ('svm', SVC(gamma='auto', probability=True)),
        ('mlp', MLPClassifier(hidden_layer_sizes=(200,100,50,),learning_rate='adaptive', early_stopping=True)),
        ('xgb', XGBClassifier(n_estimators=1000, max_depth=30, reg_alpha=0.1, reg_lambda=1.0)),
        ('lgbm', LGBMClassifier(n_estimators=1000, max_depth=30, reg_alpha=0.1, reg_lambda=1.0, min_child_samples=3, min_split_gain=0.01, verbose=-1)),
        ('catboost', CatBoostClassifier(iterations=300, learning_rate=0.1, depth=6, verbose=0))
    ]

    meta_model = LogisticRegression(max_iter=1000)
    
    stacking_model = StackingClassifier(estimators=base_learners, final_estimator=meta_model)

    # 訓練堆疊模型
    stacking_model.fit(X_train_split, y_train_split)

    # 驗證集預測
    y_val_pred = stacking_model.predict_proba(X_val_split)[:, 1]
    val_auc = roc_auc_score(y_val_split, y_val_pred)
    fold_auc_scores.append(val_auc)

    # 測試集預測
    y_test_pred = stacking_model.predict_proba(X_test)[:, 1]
    test_predictions_all_folds.append(y_test_pred)

    # 計算加權測試集預測
    total_auc = sum(fold_auc_scores)
    fold_weights = [auc / total_auc for auc in fold_auc_scores]
    test_predictions_final = np.average(test_predictions_all_folds, axis=0, weights=fold_weights)

    # 儲存測試結果 CSV
    df = pd.DataFrame(test_predictions_final, columns=['y_predict_proba'])
    df.to_csv(f'./Competition_data/{dataset_names[i]}/y_predict.csv', index=False, header=True)

    # 計算平均驗證 AUC
    avg_val_auc = np.mean(fold_auc_scores)
    print(f"Average AUC for {i+1}'th Dataset: {avg_val_auc:.4f}")
    validation_auc_scores.append(avg_val_auc)

# 儲存 AUC 分數為 CSV 文件
auc_list = pd.DataFrame(validation_auc_scores, columns=["Validation AUC"])
auc_list.to_csv('./validation_auc_scores.csv', index_label='Dataset_Index', header=True)

Average AUC for 1'th Dataset: 0.9680
Average AUC for 2'th Dataset: 1.0000
Average AUC for 3'th Dataset: 1.0000
Average AUC for 4'th Dataset: 0.9881
Average AUC for 5'th Dataset: 0.9999
Average AUC for 6'th Dataset: 0.9983
Average AUC for 7'th Dataset: 1.0000
Average AUC for 8'th Dataset: 0.9995
Average AUC for 9'th Dataset: 1.0000
Average AUC for 10'th Dataset: 0.9825
Average AUC for 11'th Dataset: 1.0000
Average AUC for 12'th Dataset: 1.0000
Average AUC for 13'th Dataset: 0.9995
Average AUC for 14'th Dataset: 1.0000
Average AUC for 15'th Dataset: 0.9929
Average AUC for 16'th Dataset: 1.0000
Average AUC for 17'th Dataset: 0.9994
Average AUC for 18'th Dataset: 1.0000
Average AUC for 19'th Dataset: 1.0000
Average AUC for 20'th Dataset: 0.9972
Average AUC for 21'th Dataset: 0.9989
Average AUC for 22'th Dataset: 0.9996
Average AUC for 23'th Dataset: 0.9998
Average AUC for 24'th Dataset: 0.9908
Average AUC for 25'th Dataset: 1.0000
Average AUC for 26'th Dataset: 0.9187
Average AUC for 27'th